# Chronos 기반 시계열 이상 탐지 (변수별)\n\n**아이디어**: 엑셀 파일의 각 변수(컬럼)를 개별 시계열로 보고, Chronos(Bolt) 모델로 \"한 스텝 앞\" 예측을 굴려가며(rolling one-step-ahead) 예측 분포(분위수)를 구한 뒤, 실제값이 예측 분포(예: 10~90% 구간) 밖으로 크게 벗어나면 그 시점 · 그 변수를 이상(anomaly)으로 표시합니다.\n\n**사용 방법**\n1. 아래 `CONFIG` 셀에서 `EXCEL_PATH`를 실제 엑셀 파일 경로로 바꿔주세요. (이 노트북과 같은 폴더에 파일을 넣으면 파일명만 적으면 됩니다)\n2. 시간 컬럼 이름(`TIME_COL`)이 있으면 적어주세요. 없으면 `None`으로 두면 행 순서를 시간축으로 사용합니다.\n3. 나머지 셀은 순서대로 실행하면 됩니다.\n\n**환경 확인 결과**: `chronos-forecasting`, `torch`, `transformers`, `pandas`, `openpyxl` 설치되어 있고, 이 Mac은 CUDA는 없지만 **MPS(Apple Silicon GPU)** 가 사용 가능합니다. 아래 코드는 자동으로 `mps` > `cuda` > `cpu` 순으로 device를 선택합니다.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from chronos import BaseChronosPipeline

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 4)


In [ ]:
# ===================== CONFIG =====================
# 파일: "운전 & 품질 데이터_2026.xlsx"
#  - 2CM/3CM/4CM 시트 = 설비 운전 데이터(근무일자+근무시간=1시간 단위 연속 센서값) -> 이번 분석 대상
#  - "2CM "/"3CM "/"4CM "(이름 끝 공백) 시트 = LIMS 품질검사 데이터(로트 단위, 불규칙) -> 이번엔 사용 안함
EXCEL_PATH = "운전 & 품질 데이터_2026.xlsx"
SHEET_NAME = "2CM"                 # 우선 2CM 하나로 검증. 나중에 "3CM", "4CM"로 바꿔서 재실행하면 됨
HEADER_ROW = 1                      # 엑셀 2번째 행이 실제 컬럼명 (0-indexed 이므로 1)

DATE_COL = "근무일자"                # 날짜 컬럼
HOUR_COL = "근무시간"                # 0~23 시간 컬럼 -> DATE_COL과 합쳐 시간축(timestamp) 생성
EXCLUDE_COLS = ["기타\n품종", "기타\n비고"]   # 범주형/텍스트 컬럼은 이상탐지 대상에서 제외

CONTEXT_LENGTH = 168                # 예측에 사용할 과거 구간 길이 (168시간 = 1주일)
PREDICTION_LENGTH = 1               # 한 번에 몇 스텝(시간) 앞을 예측할지
STRIDE = 1                          # 몇 스텝마다 예측을 수행할지 (1 = 매 시점마다)
QUANTILE_LEVELS = [0.1, 0.5, 0.9]    # 예측 분포에서 사용할 분위수 (하한, 중앙값, 상한)
ANOMALY_QUANTILE_LOW = 0.1
ANOMALY_QUANTILE_HIGH = 0.9

MODEL_ID = "amazon/chronos-bolt-small"   # 필요시 amazon/chronos-bolt-base 등으로 변경 가능
BATCH_SIZE = 256                          # 한 번에 모델에 넣을 윈도우 개수 (메모리에 맞게 조절)

# 41개 변수 x 약 4만 시점 x STRIDE=1 전체를 다 돌리면 M-series MPS 기준 약 1시간 소요됩니다.
# 먼저 빠르게 검증하고 싶으면 아래 값을 사용하세요 (검증 끝나면 None으로 바꿔서 전체 재실행).
QUICK_TEST_ROWS = 5000    # 최근 N행(시간)만 사용. 전체 데이터로 돌리려면 None

# device 자동 선택: mps(Apple GPU) > cuda > cpu
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Selected device: {DEVICE}")


In [ ]:
# ===================== 데이터 로드 =====================
def load_data(path, sheet_name, header_row, date_col, hour_col, exclude_cols):
    df = pd.read_excel(path, sheet_name=sheet_name, header=header_row)

    time_index = pd.to_datetime(df[date_col]) + pd.to_timedelta(df[hour_col], unit="h")
    order = time_index.argsort()
    time_index = time_index.iloc[order].reset_index(drop=True)
    df = df.iloc[order].reset_index(drop=True)

    df = df.drop(columns=[c for c in [date_col, hour_col, *exclude_cols] if c in df.columns])

    # 숫자형 컬럼만 이상탐지 대상으로 사용
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    non_numeric = [c for c in df.columns if c not in numeric_cols]
    if non_numeric:
        print(f"숫자형이 아니라 제외된 컬럼: {non_numeric}")

    data = df[numeric_cols].copy()

    # 설비 미가동 등으로 인한 결측치는 직전/직후 값으로 채움 (ffill -> bfill)
    n_missing = data.isna().sum().sum()
    if n_missing:
        print(f"결측치 {n_missing}개 -> ffill/bfill로 채움")
        data = data.ffill().bfill()

    return data, time_index, numeric_cols


data, time_index, variable_cols = load_data(EXCEL_PATH, SHEET_NAME, HEADER_ROW, DATE_COL, HOUR_COL, EXCLUDE_COLS)

if QUICK_TEST_ROWS is not None:
    data = data.tail(QUICK_TEST_ROWS).reset_index(drop=True)
    time_index = time_index.tail(QUICK_TEST_ROWS).reset_index(drop=True)
    print(f"QUICK_TEST_ROWS={QUICK_TEST_ROWS} 적용 -> 최근 {len(data)}행만 사용 (검증 끝나면 CONFIG에서 None으로 변경 후 재실행)")

print(f"행 수: {len(data)}, 변수 개수: {len(variable_cols)}")
print(f"기간: {time_index.min()} ~ {time_index.max()}")
print(f"변수 목록: {variable_cols}")
data.head()


In [ ]:
# ===================== Chronos 모델 로드 =====================
pipeline = BaseChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.float32,
)
print(f"Loaded {MODEL_ID} on {DEVICE}")


In [ ]:
# ===================== 변수별 rolling one-step-ahead 예측 =====================
# 각 변수(컬럼)에 대해: 과거 CONTEXT_LENGTH 구간을 보고 다음 시점을 예측 -> 실제값과 비교
# 예측 분위수 구간(q_low~q_high) 밖으로 실제값이 벗어나면 그 시점을 이상치로 표시합니다.

low_q, mid_q, high_q = ANOMALY_QUANTILE_LOW, 0.5, ANOMALY_QUANTILE_HIGH
assert low_q in QUANTILE_LEVELS and mid_q in QUANTILE_LEVELS and high_q in QUANTILE_LEVELS, \
    "QUANTILE_LEVELS에 low/median/high 분위수가 모두 포함되어야 합니다."
low_i, mid_i, high_i = QUANTILE_LEVELS.index(low_q), QUANTILE_LEVELS.index(mid_q), QUANTILE_LEVELS.index(high_q)


def rolling_forecast(values: np.ndarray) -> pd.DataFrame:
    n = len(values)
    positions = list(range(CONTEXT_LENGTH, n, STRIDE))
    if not positions:
        raise ValueError(
            f"데이터 길이({n})가 CONTEXT_LENGTH({CONTEXT_LENGTH})보다 짧습니다. CONTEXT_LENGTH를 줄여주세요."
        )

    rows = []
    for b in range(0, len(positions), BATCH_SIZE):
        batch_pos = positions[b : b + BATCH_SIZE]
        contexts = [torch.tensor(values[i - CONTEXT_LENGTH : i], dtype=torch.float32) for i in batch_pos]

        quantiles, _mean = pipeline.predict_quantiles(
            inputs=contexts,
            prediction_length=PREDICTION_LENGTH,
            quantile_levels=QUANTILE_LEVELS,
        )
        quantiles = quantiles.detach().to("cpu").float().numpy()  # [batch, prediction_length, num_quantiles]

        for j, i in enumerate(batch_pos):
            actual = float(values[i])  # PREDICTION_LENGTH 중 첫 번째 스텝(다음 시점)과 비교
            q_low = float(quantiles[j, 0, low_i])
            q_mid = float(quantiles[j, 0, mid_i])
            q_high = float(quantiles[j, 0, high_i])
            band = max(q_high - q_low, 1e-8)
            is_anomaly = actual < q_low or actual > q_high
            severity = 0.0
            if actual < q_low:
                severity = (q_low - actual) / band
            elif actual > q_high:
                severity = (actual - q_high) / band

            rows.append(
                dict(
                    idx=i,
                    actual=actual,
                    pred_median=q_mid,
                    pred_low=q_low,
                    pred_high=q_high,
                    error=actual - q_mid,
                    is_anomaly=is_anomaly,
                    severity=severity,
                )
            )

    return pd.DataFrame(rows)


In [ ]:
# ===================== 모든 변수에 대해 실행 =====================
results = {}  # 변수명 -> 결과 DataFrame

for col in variable_cols:
    print(f"처리 중: {col}")
    series_values = data[col].to_numpy(dtype=np.float32)
    res = rolling_forecast(series_values)
    res["timestamp"] = [time_index[i] for i in res["idx"]]
    res["variable"] = col
    results[col] = res

all_results = pd.concat(results.values(), ignore_index=True)
print(f"\n총 {len(all_results)}개 예측, 이상치 {all_results['is_anomaly'].sum()}개 탐지됨")
all_results.head()


In [ ]:
# ===================== 변수별 이상치 요약 =====================
summary = (
    all_results.groupby("variable")
    .agg(n_points=("is_anomaly", "size"), n_anomalies=("is_anomaly", "sum"), max_severity=("severity", "max"))
    .assign(anomaly_rate=lambda d: d["n_anomalies"] / d["n_points"])
    .sort_values("n_anomalies", ascending=False)
)
summary


In [ ]:
# ===================== 변수별 시각화 (실제값 vs 예측 구간, 이상치 표시) =====================
def plot_variable(col):
    res = results[col]
    fig, ax = plt.subplots()
    ax.plot(res["timestamp"], res["actual"], label="actual", color="tab:blue", linewidth=1)
    ax.plot(res["timestamp"], res["pred_median"], label="predicted (median)", color="tab:orange", linewidth=1)
    ax.fill_between(res["timestamp"], res["pred_low"], res["pred_high"], color="tab:orange", alpha=0.2, label="pred band")

    anomalies = res[res["is_anomaly"]]
    ax.scatter(anomalies["timestamp"], anomalies["actual"], color="red", s=20, zorder=5, label="anomaly")

    ax.set_title(f"{col} — actual vs Chronos prediction ({len(anomalies)} anomalies)")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


# 이상치가 많은 변수부터 확인 (원하는 변수명으로 바꿔서 호출해도 됩니다)
for col in summary.index[:5]:
    plot_variable(col)


In [ ]:
# ===================== 결과 저장 =====================
OUTPUT_PATH = "chronos_anomaly_results.csv"
cols_to_save = ["variable", "timestamp", "actual", "pred_median", "pred_low", "pred_high", "error", "severity", "is_anomaly"]
all_results[cols_to_save].to_csv(OUTPUT_PATH, index=False)
print(f"저장 완료: {OUTPUT_PATH}")

# 이상치만 모아서 보기
all_results[all_results["is_anomaly"]][cols_to_save].sort_values("severity", ascending=False)
